# 7 · Mixed problems — the saddle point 🐎

![A pirate balancing a saddle with "velocity u" and "pressure p" bags atop a peak — Unit 7, the saddle point](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/saddle.jpg)

Welcome **into the saddle**. Part II opens with its mathematical namesake: the
**saddle-point problem**.

The classic example is **Stokes flow** — slow, viscous, incompressible fluid
( for mixed Poisson see [i-tutorials](https://docu.ngsolve.org/latest/i-tutorials/unit-2.5-mixed/mixed.html) )
:
$$ -\Delta \mathbf u + \nabla p = \mathbf f,\qquad \operatorname{div}\mathbf u = 0 . $$
* The velocity $\mathbf u$ wants to minimise the viscous energy,
* but is **constrained** to be divergence-free (incompressible);
* the **pressure** $p$ is exactly the multiplier enforcing that constraint.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
from netgen.occ import unit_square
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.08))

## 1. One product space — with a loose pressure constant

Discretise velocity and pressure **together** in a **product space** $X=V\times Q$ — the
**Taylor–Hood** pair: vector $H^1$ of order $k$ for $\mathbf u$, scalar $H^1$ of order $k-1$
for $p$ (this pairing is *inf-sup stable*: it does not lock). The whole Stokes weak form is
then **one** bilinear form on $X$,
$$ \int \nabla\mathbf u\!:\!\nabla\mathbf v + \operatorname{div}\mathbf v\,p + \operatorname{div}\mathbf u\,q \;=\; \int\mathbf f\!\cdot\!\mathbf v, $$
assembling into $K=\bigl(\begin{smallmatrix}A&B^\top\\B&0\end{smallmatrix}\bigr)$ — the
saddle-point signature, with a **zero pressure block**. 

In [ ]:
V = VectorH1(mesh, order=2, dirichlet="bottom|right|top|left")   # velocity
Q = H1(mesh, order=1)                                            # pressure
X = V * Q                                                        # the product space
(u, p), (v, q) = X.TnT()
stokes = (InnerProduct(Grad(u), Grad(v)) + div(u) * q + div(v) * p) * dx
print(f"product space: {X.ndof} scalar dofs  ({V.ndof} scalar velocity + {Q.ndof} pressure)")

## 2. Pinning the pressure

The loose constant can be pinned in several ways; they differ in **which solver** each then
allows. We **solve with the first** and construct the others to show the interface:

1. **Regularisation** $-\varepsilon\!\int p\,q$ — fills the zero pivot, so `sparsecholesky`
   factors the whole system; $\varepsilon$ only slightly relaxes incompressibility. ← used here.
2. **Pin one pressure dof** ($p=0$ at a node) — clear one bit of `FreeDofs`. Removes the
   nullspace but leaves $K$ **indefinite**, so it needs an indefinite solver (`umfpack`, or the
   block `MinRes` of §4).
3. **A `NumberSpace` multiplier** ($\int p=0$) — the cleanest constraint, `X = V*Q*NumberSpace`;
   also indefinite → block / iterative solver.
4. **Subtract the mean** afterwards — post-hoc, gives the canonical $\int p=0$ (done in §3).

In [ ]:
eps = 1e-8
a = BilinearForm(X)
a += stokes - eps * p * q * dx                     # (1) regularisation — keeps a direct Cholesky
a.Assemble()

free_pinned = BitArray(X.FreeDofs()); free_pinned.Clear(V.ndof)   # (2) pin the first pressure dof
XN = V * Q * NumberSpace(mesh)                                    # (3) global ∫p=0 multiplier
print(f"(1) regularised K assembled · (2) pinned freedofs keep {free_pinned.NumSet()} of {X.ndof} · "
      f"(3) NumberSpace-augmented space: {XN.ndof} dofs")

## 3. Solve it — a lid-driven cavity

The box is closed; we drag the **lid** (top edge) tangentially, with a profile fading to zero
at the corners. Set the boundary velocity, move its effect to the right-hand side (the lifting
of unit 5), and solve the **whole** regularised system in one direct `sparsecholesky`. Then
subtract the pressure mean for the canonical $\int p=0$ (pin #4). The lid drags the fluid into
one big **recirculating vortex**; the pressure multiplier adjusts to keep the flow div-free.

In [ ]:
gf = GridFunction(X)
lid = CF((16 * x * x * (1 - x) * (1 - x), 0))          # 1 in the middle, 0 at the corners
gf.components[0].Set(lid, definedon=mesh.Boundaries("top"))
res = -a.mat * gf.vec
gf.vec.data += a.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky") * res

velocity, pressure = gf.components
pressure0 = GridFunction(Q)
pressure0.Set(pressure - Integrate(pressure, mesh))     # (4) canonical zero-mean pressure

In [ ]:
Draw(velocity, mesh, "velocity", vectors={"grid_size": 26})

In [ ]:
Draw(pressure0, mesh, "pressure p (the constraint's multiplier)")

## 4. Block solver — `MinRes`

The **scalable** alternative to the direct solver keeps the blocks **separate**: 
* assemble $K$ as a `BlockMatrix` 
* and solve with **`MinRes`**
* preconditioned **block-diagonally** by the two pieces that *are* SPD:
    * the velocity stiffness $A^{-1}$
    * and the **pressure mass matrix** $M_p^{-1}$ 

In [ ]:
u_, v_ = V.TnT(); p_, q_ = Q.TnT()
Amat = BilinearForm(InnerProduct(Grad(u_), Grad(v_)) * dx).Assemble()
Bmat = BilinearForm(trialspace=V, testspace=Q); Bmat += div(u_) * q_ * dx; Bmat.Assemble()
Mp   = BilinearForm(p_ * q_ * dx).Assemble()                          # pressure mass matrix

# BlockMatrix: BaseMatrix (linear operator; without dedicated memory)
K = BlockMatrix([[Amat.mat, Bmat.mat.T], [Bmat.mat, None]])           # the saddle operator
C = BlockMatrix([[Amat.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky"), None],
                 [None, Mp.mat.Inverse(inverse="sparsecholesky")]])   # block preconditioner

gu = GridFunction(V); gp = GridFunction(Q)
gu.Set(lid, definedon=mesh.Boundaries("top"))
rhs = BlockVector([-Amat.mat * gu.vec, -Bmat.mat * gu.vec])
du = gu.vec.CreateVector(); du[:] = 0
dp = gp.vec.CreateVector(); dp[:] = 0
with TaskManager():
    solvers.MinRes(mat=K, pre=C, rhs=rhs, sol=BlockVector([du, dp]),
                   maxsteps=500, tol=1e-10, printrates=False)
gu.vec.data += du; 
gp.vec.data += dp
gp0 = GridFunction(Q)
gp0.Set(gp - Integrate(gp, mesh))

In [ ]:
Draw(gu, mesh, "velocity", vectors={"grid_size": 26})

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("06-linear-solvers", "6 · The solver toolbox 🛠")
    _next = ("08-unsteady-doubleglazing", "8 · Unsteady problems — the double-glazing flow")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))